# 02 · Node Embeddings

## What this notebook covers
Before GNNs, graph ML used hand-crafted node embeddings from random walks. This notebook covers the historical context and implements Node2Vec — which is still useful for graphs where GNNs are overkill.

## DeepWalk and Node2Vec — historical context
**DeepWalk** (2014) was the first major graph embedding method. It runs random walks on the graph and feeds the resulting node sequences into Word2Vec's skip-gram objective. Nodes that co-occur in random walks get similar embeddings. It captures global community structure well.

**Node2Vec** (2016) extended DeepWalk with biased random walks controlled by two parameters:
- **p** (return parameter): controls likelihood of returning to visited nodes (low p = DFS-like, captures communities)
- **q** (in-out parameter): controls likelihood of exploring outward (low q = BFS-like, captures structural roles)

**Why they lost to GNNs**: DeepWalk and Node2Vec are *transductive* — they can't embed nodes not seen during training. GNNs are *inductive* via learnable aggregation. For most modern tasks, GNNs are preferred. Node2Vec remains useful for:
- Graphs too large to train GNNs on (billions of edges)
- Unsupervised pre-training when labels are unavailable
- Quick baseline embeddings

In [ ]:
import numpy as np
import networkx as nx
from sklearn.manifold import TSNE
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)

# Karate club: the classic graph ML benchmark
G = nx.karate_club_graph()
labels = {n: 0 if G.nodes[n]["club"] == "Mr. Hi" else 1 for n in G.nodes()}
y = np.array([labels[n] for n in G.nodes()])
print(f"Karate club graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges, 2 communities")

In [ ]:
# ── Node2Vec biased random walks ───────────────────────────────────────────────
def biased_random_walk(G, start, walk_length, p=1.0, q=1.0):
    walk = [start]
    while len(walk) < walk_length:
        cur = walk[-1]
        neighbours = list(G.neighbors(cur))
        if not neighbours:
            break
        if len(walk) == 1:
            walk.append(np.random.choice(neighbours))
        else:
            prev = walk[-2]
            probs = []
            for nb in neighbours:
                if nb == prev:
                    probs.append(1/p)
                elif G.has_edge(prev, nb):
                    probs.append(1.0)
                else:
                    probs.append(1/q)
            probs = np.array(probs)
            probs /= probs.sum()
            walk.append(np.random.choice(neighbours, p=probs))
    return walk

def node2vec_walks(G, num_walks=20, walk_length=40, p=1.0, q=1.0):
    walks = []
    for _ in range(num_walks):
        for node in G.nodes():
            walks.append(biased_random_walk(G, node, walk_length, p, q))
    return walks

walks = node2vec_walks(G, p=0.5, q=2.0)
print(f"Generated {len(walks)} walks")
print(f"Example walk: {walks[0][:10]}...")

In [ ]:
# ── Skip-gram training (simplified Word2Vec) ──────────────────────────────────
from collections import defaultdict

def train_skipgram(walks, nodes, embedding_dim=32, window=5, epochs=5, lr=0.025):
    n = len(nodes)
    node_idx = {node: i for i, node in enumerate(nodes)}
    W  = np.random.randn(n, embedding_dim) * 0.01  # input embeddings
    W2 = np.random.randn(embedding_dim, n) * 0.01  # output embeddings

    for epoch in range(epochs):
        total_loss = 0
        for walk in walks:
            for i, center in enumerate(walk):
                context = walk[max(0, i-window):i] + walk[i+1:i+window+1]
                for ctx in context:
                    ci, cxi = node_idx[center], node_idx[ctx]
                    h = W[ci]                        # (D,)
                    scores = W2.T @ h                 # (N,)
                    probs  = np.exp(scores - scores.max())
                    probs /= probs.sum()
                    loss = -np.log(probs[cxi] + 1e-9)
                    total_loss += loss
                    # Gradients
                    dscores = probs.copy(); dscores[cxi] -= 1
                    W2  -= lr * np.outer(h, dscores)
                    W[ci] -= lr * (W2 @ dscores)
    return W

nodes = list(G.nodes())
embeddings = train_skipgram(walks, nodes, embedding_dim=16, epochs=3)
print(f"Learned embeddings: {embeddings.shape}")

# Downstream: node classification
X_emb = embeddings
cv_scores = cross_val_score(LogisticRegression(max_iter=200), X_emb, y, cv=5, scoring="accuracy")
print(f"Node classification accuracy (5-fold): {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

In [ ]:
# Visualise embeddings
tsne = TSNE(n_components=2, random_state=0, perplexity=5)
emb_2d = tsne.fit_transform(embeddings)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
colours = ["steelblue", "tomato"]
axes[0].scatter(emb_2d[:, 0], emb_2d[:, 1], c=[colours[yi] for yi in y], s=80, alpha=0.8)
for i, node in enumerate(nodes):
    axes[0].annotate(str(node), emb_2d[i], fontsize=6, alpha=0.6)
axes[0].set_title("Node2Vec embeddings (t-SNE)")

pos = nx.spring_layout(G, seed=42)
nx.draw(G, pos=pos, ax=axes[1], node_color=[colours[labels[n]] for n in G.nodes()],
        with_labels=True, font_size=7, node_size=200, edge_color="grey", alpha=0.8)
axes[1].set_title("Karate club graph (blue=Mr.Hi, red=Officer)")

plt.tight_layout()
plt.savefig(f"{base}/02_node_embeddings.png", dpi=120)
plt.show()